In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

df = pd.read_csv("force2020_cleaned.csv")
curves = ["RHOB", "GR", "NPHI", "PEF", "DTC"]
units = {"RHOB": "g/cc", "GR": "API", "NPHI": "v/v", "PEF": "b/e", "DTC": "us/ft"}
flags = [col for col in df.columns if col.startswith("flag_")]
step = df["DEPTH_MD"].diff().median()

print("Rows:", len(df), "| step:", round(step, 3), "m")
print("Flags:", flags)
print("Rows with all 5 curves:", int(df[curves].notna().all(axis=1).sum()))


Rows: 18270 | step: 0.152 m
Flags: ['flag_bad_hole', 'flag_pef_high', 'flag_pef_spike', 'flag_dtc_frozen', 'flag_gr_high']
Rows with all 5 curves: 12018


In [3]:
def build_features(centre="median", window_m=4, spread=None, cols=None):
    cols = cols or curves
    n = int(round(window_m / step)) | 1
    out = {}
    for col in cols:
        roll = df[col].rolling(window=n, center=True, min_periods=n // 2 + 1)
        out[f"{col}_c"] = roll.median() if centre == "median" else roll.mean()
        if spread == "std":
            out[f"{col}_s"] = roll.std()
        elif spread == "mad":
            out[f"{col}_s"] = roll.apply(lambda v: np.nanmedian(np.abs(v - np.nanmedian(v))), raw=True)
    return pd.DataFrame(out, index=df.index)


def score(features, rows, name, baseline=None, k=4):
    X = pd.DataFrame(RobustScaler().fit_transform(features[rows]),
                     index=features.index[rows], columns=features.columns)
    labels = pd.Series(KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X), index=X.index)
    result = {"variant": name, "features": X.shape[1], "rows": len(X),
              "silhouette": round(silhouette_score(X, labels, sample_size=5000, random_state=42), 3),
              "continuity": round((labels.to_numpy()[1:] == labels.to_numpy()[:-1]).mean(), 4)}
    if baseline is not None:
        result["agrees with baseline"] = round(adjusted_rand_score(baseline, labels), 3)
    return result, labels


In [4]:
variants = {
    "median": ("median", None),
    "mean": ("mean", None),
    "median + MAD": ("median", "mad"),
    "mean + std": ("mean", "std"),
}

built = {name: build_features(centre, window_m=4, spread=spread) for name, (centre, spread) in variants.items()}

rows = pd.Series(True, index=df.index)
for table in built.values():
    rows &= table.notna().all(axis=1)
print("Rows used by every variant:", int(rows.sum()))

baseline = None
results = []
for name, table in built.items():
    result, labels = score(table, rows, name, baseline=baseline)
    if baseline is None:
        baseline = labels
    results.append(result)

test_a = pd.DataFrame(results).set_index("variant")
test_a


Rows used by every variant: 12104


,features,rows,silhouette,continuity,agrees with baseline
variant,,,,,
median,5,12104,0.546,0.9953,NaN
mean,5,12104,0.531,0.9960,0.931
median + MAD,10,12104,0.301,0.9820,0.718
mean + std,10,12104,0.309,0.9880,0.624


In [5]:
windows = [2, 4, 6, 8]
built_b = {f"{w} m": build_features("median", window_m=w) for w in windows}

rows_b = pd.Series(True, index=df.index)
for table in built_b.values():
    rows_b &= table.notna().all(axis=1)
print("Rows used by every window:", int(rows_b.sum()))

_, baseline_4m = score(built_b["4 m"], rows_b, "4 m")

results = [score(table, rows_b, name, baseline=baseline_4m)[0] for name, table in built_b.items()]
test_b = pd.DataFrame(results).set_index("variant")
test_b


Rows used by every window: 12085


,features,rows,silhouette,continuity,agrees with baseline
variant,,,,,
2 m,5,12085,0.556,0.9928,0.924
4 m,5,12085,0.550,0.9953,1.000
6 m,5,12085,0.549,0.9968,0.960
8 m,5,12085,0.551,0.9972,0.948


In [6]:
features_4m = build_features("median", window_m=4)
features_no_dtc = build_features("median", window_m=4, cols=[c for c in curves if c != "DTC"])

rows_all = features_4m.notna().all(axis=1)
rows_unfrozen = rows_all & ~df["flag_dtc_frozen"]

_, baseline_labels = score(features_4m, rows_all, "all rows")

results = []
for name, table, rows in [("all rows", features_4m, rows_all),
                          ("frozen DTC rows dropped", features_4m, rows_unfrozen),
                          ("DTC curve dropped", features_no_dtc, rows_all)]:
    result, labels = score(table, rows, name)
    shared = labels.index.intersection(baseline_labels.index)
    result["agrees with baseline"] = round(adjusted_rand_score(baseline_labels[shared], labels[shared]), 3)
    results.append(result)

test_c = pd.DataFrame(results).set_index("variant")
test_c


,features,rows,silhouette,continuity,agrees with baseline
variant,,,,,
all rows,5,12104,0.546,0.9953,1.000
frozen DTC rows dropped,5,9867,0.513,0.9948,0.915
DTC curve dropped,4,12104,0.551,0.9959,0.958


In [7]:
features_4m = build_features("median", window_m=4)
rows_4m = features_4m.notna().all(axis=1)
X = pd.DataFrame(RobustScaler().fit_transform(features_4m[rows_4m]),
                 index=features_4m.index[rows_4m], columns=features_4m.columns)

results = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    labels = pd.Series(km.labels_, index=X.index)
    results.append({"k": k, "inertia": round(km.inertia_, 1),
                    "silhouette": round(silhouette_score(X, labels, sample_size=5000, random_state=42), 3),
                    "continuity": round((labels.to_numpy()[1:] == labels.to_numpy()[:-1]).mean(), 4),
                    "smallest cluster %": round(100 * labels.value_counts().min() / len(labels), 1)})

test_d = pd.DataFrame(results).set_index("k")
test_d


,inertia,silhouette,continuity,smallest cluster %
k,,,,
2,14927.4,0.515,0.9988,31.6
3,9715.4,0.576,0.9987,13.2
4,6706.5,0.546,0.9953,12.6
5,5236.4,0.557,0.9951,1.1
6,4038.7,0.505,0.9943,1.1
7,3475.6,0.481,0.9943,1.1
8,3028.1,0.485,0.9931,1.1


In [8]:
raw = pd.read_csv("force2020_data_unsupervised_learning.csv")

spike = df["flag_pef_spike"]
pef_ffill = df["PEF"].copy()
pef_ffill[spike] = np.nan
pef_ffill = pef_ffill.ffill()

comparison = pd.DataFrame({"raw spike": raw.loc[spike, "PEF"],
                           "median fix": df.loc[spike, "PEF"],
                           "ffill fix": pef_ffill[spike]})
print(comparison.describe().round(2).loc[["mean", "50%", "min", "max"]])
print("median difference between the fixes: %.3f" % (comparison["median fix"] - comparison["ffill fix"]).abs().median())

features_ffill = build_features("median", window_m=4)
features_ffill["PEF_c"] = pef_ffill.rolling(27, center=True, min_periods=14).median()

rows_f = features_4m.notna().all(axis=1) & features_ffill.notna().all(axis=1)
_, labels_median = score(features_4m, rows_f, "PEF spikes: 1 m median")
pd.DataFrame([score(features_4m, rows_f, "PEF spikes: 1 m median")[0],
              score(features_ffill, rows_f, "PEF spikes: forward fill", baseline=labels_median)[0]]).set_index("variant")


      raw spike  median fix  ffill fix
mean       6.37        5.74       5.90
50%        6.38        5.65       5.95
min        2.85        2.83       3.12
max        9.77        9.09       9.79
median difference between the fixes: 0.343


,features,rows,silhouette,continuity,agrees with baseline
variant,,,,,
PEF spikes: 1 m median,5,12104,0.546,0.9953,NaN
PEF spikes: forward fill,5,12104,0.549,0.9955,0.99


In [9]:
from sklearn.cluster import HDBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from scipy.sparse import diags_array

X = pd.DataFrame(RobustScaler().fit_transform(features_4m[rows_4m]),
                 index=features_4m.index[rows_4m], columns=features_4m.columns)
depth = df.loc[rows_4m, "DEPTH_MD"]
chain = diags_array([np.ones(len(X) - 1), np.ones(len(X) - 1)], offsets=[-1, 1], shape=(len(X), len(X)))

def metrics(labels):
    lab = pd.Series(labels, index=X.index)
    keep = lab != -1
    return (round(silhouette_score(X[keep], lab[keep], sample_size=5000, random_state=42), 3),
            round((lab.to_numpy()[1:] == lab.to_numpy()[:-1]).mean(), 4), lab)

results, labels_by_run = [], {}
for k in (3, 4):
    runs = {"kmeans": KMeans(k, n_init=10, random_state=42).fit_predict(X),
            "gmm": GaussianMixture(k, n_init=3, random_state=42).fit(X).predict(X),
            "agglo": AgglomerativeClustering(n_clusters=k, linkage="ward", connectivity=chain).fit(X).labels_}
    for name, raw_labels in runs.items():
        sil, cont, lab = metrics(raw_labels)
        labels_by_run[(k, name)] = lab
        boundaries = ", ".join(f"{d:.0f}" for d in depth[lab.ne(lab.shift()).to_numpy()].iloc[1:]) if name == "agglo" else ""
        results.append({"k": k, "method": name, "silhouette": sil, "continuity": cont, "boundaries": boundaries})
    pairs = [("kmeans", "gmm"), ("kmeans", "agglo"), ("gmm", "agglo")]
    agree = np.mean([adjusted_rand_score(labels_by_run[(k, a)], labels_by_run[(k, b)]) for a, b in pairs])
    results.append({"k": k, "method": "mean agreement", "silhouette": round(agree, 3), "continuity": "", "boundaries": ""})

hdb = HDBSCAN(min_cluster_size=200, copy=True).fit(X)
sil, cont, lab = metrics(hdb.labels_)
print("HDBSCAN (picks its own k):", lab[lab != -1].nunique(), "clusters,", int((lab == -1).sum()),
      "noise, silhouette", sil, "continuity", cont)

pd.DataFrame(results)


HDBSCAN (picks its own k): 3 clusters, 1102 noise, silhouette 0.605 continuity 0.9926


,k,method,silhouette,continuity,boundaries
0,3,kmeans,0.576,0.9987,
1,3,gmm,0.379,0.996,
2,3,agglo,0.555,0.9998,"2395, 2741"
3,3,mean agreement,0.645,,
4,4,kmeans,0.546,0.9953,
5,4,gmm,0.439,0.9951,
6,4,agglo,0.461,0.9998,"1997, 2395, 2741"
7,4,mean agreement,0.713,,


In [10]:
for k in (3, 4):
    labels = labels_by_run[(k, "kmeans")]
    profile = df.loc[rows_4m].groupby(labels)[curves].median().round(2)
    profile["rows"] = labels.value_counts().sort_index()
    profile["top"] = df.loc[rows_4m].groupby(labels)["DEPTH_MD"].min().round(0)
    profile["bottom"] = df.loc[rows_4m].groupby(labels)["DEPTH_MD"].max().round(0)
    profile["median depth"] = df.loc[rows_4m].groupby(labels)["DEPTH_MD"].median().round(0)
    print(f"\nk = {k}")
    print(profile.to_string())



k = 3
   RHOB     GR  NPHI   PEF     DTC  rows     top  bottom  median depth
0  2.01  65.20  0.50  2.80  144.65  8184  1139.0  2439.0        1776.0
1  2.54  17.57  0.18  4.70   71.26  2328  2234.0  2790.0        2575.0
2  2.48  96.21  0.31  4.41   88.49  1592  2732.0  2994.0        2873.0

k = 4
   RHOB     GR  NPHI   PEF     DTC  rows     top  bottom  median depth
0  2.02  69.21  0.50  2.67  145.69  6591  1139.0  2329.0        1665.0
1  2.54  17.17  0.17  4.70   70.44  2279  2396.0  2790.0        2575.0
2  2.02  43.61  0.50  5.14  125.51  1703  1189.0  2770.0        2224.0
3  2.48  96.61  0.31  4.38   88.45  1531  2740.0  2994.0        2878.0


### Feature testing — conclusions

| Test | Options | Winner |
|---|---|---|
| A: centre and spread | median / mean / median+MAD / mean+std | **median** (0.546 vs 0.531; spread columns halved the score) |
| B: window | none / 2 / 4 / 6 / 8 m | **4 m** — scores flat (0.546–0.556), all agree 0.92+; smoothing mainly buys continuity (0.975 raw → 0.996) |
| C: DTC | all rows / drop frozen rows / drop DTC | **keep everything** — dropping the 4,846 frozen rows changes 8.5% of the grouping and costs 2,237 depths |
| D: number of clusters | k = 2–8 | **k = 3** (0.576) — k = 4 is 0.546; k ≥ 5 produces 1.1% slivers |
| Stage 2: four methods | k = 3 vs k = 4 | **k = 3** — HDBSCAN also picks 3 on its own; the k = 4 extra group is scattered over 1189–2770 m, not a rock unit |
| Extra: spike repair | 1 m median vs forward fill | either — 99% identical clusters, 0.002 silhouette apart |

**Final configuration for the model**
- data: `force2020_cleaned.csv`
- features: 4 m rolling **median** of RHOB, GR, NPHI, PEF, DTC (27 rows, centred, min 14)
- scaling: RobustScaler
- rows: all depths with five curves, **12,104** (1138.7–2993.9 m)
- **k = 3**, KMeans as the primary method

**Result to carry forward**
- ~1139–2439 m: soft shale (RHOB 2.01, GR 65, NPHI 0.50, PEF 2.8, DTC 145)
- ~2395–2790 m: limestone / chalk (2.54, 18, 0.18, 4.7, 71)
- ~2741–2994 m: compacted shale (2.48, 96, 0.31, 4.4, 88)
- boundaries ~2395 m and ~2741 m appear in every method and survive cleaning
- the upper shale subdivides at 1997 m on PEF (2.6 → 4.0) — a mineral change, not a separate lithology
